# SegmentAE Example: Built-in Autoencoder Models

This notebook demonstrates how to use SegmentAE with built-in DenseAutoencoder for anomaly detection with comprehensive performance evaluation.

## Features demonstrated:
- Built-in DenseAutoencoder architecture
- Type-safe configuration options
- Enhanced error handling and validation
- Multiple threshold evaluation
- Comprehensive performance metrics

## Import Dependencies

In [ ]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

from segmentae import Clustering, SegmentAE
from segmentae.autoencoders import DenseAutoencoder  # , BatchNormAutoencoder
from segmentae.data_sources import load_dataset
from segmentae.metrics import metrics_classification
from segmentae.processing.preprocessing import Preprocessing

## Data Loading

Load the HTRU2 pulsar dataset and split into train/test/future sets.

In [ ]:
train, test, target = load_dataset(
    dataset_selection="htru2_dataset",  # Other Options Available
    split_ratio=0.75,
)

test, future_data = train_test_split(test, train_size=0.9, random_state=5)

# Resetting the index is optional: rows are handled by position and results
# keep the index of the data they were computed from
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)
future_data = future_data.reset_index(drop=True)

X_train = train.drop(columns=[target]).copy()
X_test, y_test = test.drop(columns=[target]).copy(), test[target].astype(int)
X_future_data = future_data.drop(columns=[target]).copy()
y_future_data = future_data[target].astype(int)

print("Dataset loaded successfully!")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print(f"  Future samples: {len(X_future_data)}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Target distribution: {train[target].value_counts().to_dict()}")

## Data Preprocessing

Apply feature encoding, scaling, and optional imputation.

In [ ]:
# MinMaxScaler maps the features to [0, 1], the range a sigmoid output layer can
# reconstruct. With "StandardScaler" or "RobustScaler", use a linear output layer instead.
pr = Preprocessing(
    encoder="IFrequencyEncoder",  # "LabelEncoder", "OneHotEncoder"
    scaler="MinMaxScaler",  # "StandardScaler", "RobustScaler"
    imputer=None,
)

pr.fit(X=X_train)
X_train = pr.transform(X=X_train)
X_test = pr.transform(X=X_test)
X_future_data = pr.transform(X=X_future_data)

print("✓ Data preprocessing completed")
print(f"  Categorical columns: {pr.cat_cols}")
print(f"  Numerical columns: {len(pr.num_cols)} columns")

## Clustering Implementation

Set up clustering with the registry pattern for easy extensibility.

In [ ]:
cl_model = Clustering(
    cluster_model=["KMeans"],  # "KMeans", "MiniBatchKMeans", "GMM"
    n_clusters=3,
)
cl_model.clustering_fit(X=X_train)

print("✓ Clustering completed")
print(f"  Method: {cl_model.cluster_model}")
print(f"  Number of clusters: {cl_model.n_clusters}")

## Built-in Autoencoder Implementation

Train a DenseAutoencoder with comprehensive configuration options.

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

denseAutoencoder = DenseAutoencoder(
    hidden_dims=[16, 8, 4],
    encoder_activation="relu",
    decoder_activation="relu",
    output_activation="sigmoid",  # matches MinMax-scaled data
    optimizer="adam",
    learning_rate=0.001,
    epochs=150,
    val_size=0.15,
    stopping_patient=10,  # Early stopping
    dropout_rate=0.1,
    batch_size=None,
    random_state=42,
)

print("Training autoencoder...")
denseAutoencoder.fit(input_data=X_train)
print("✓ Autoencoder training completed")

## Model Architecture and Training Visualization

In [ ]:
denseAutoencoder.summary()
denseAutoencoder.plot_training_loss()

## SegmentAE Integration

Combine the autoencoder and clustering for anomaly detection.

In [ ]:
sg = SegmentAE(ae_model=denseAutoencoder, cl_model=cl_model)

# Train Reconstruction with Metrics
sg.reconstruction(
    input_data=X_train,
    threshold_metric="mse",  # "mse", "mae", "rmse", "max_error"
)

print("✓ Reconstruction phase completed")
print("  Baseline threshold per cluster (mean training reconstruction error):")
for cluster_id, threshold in sg.thresholds.items():
    print(f"    Cluster {cluster_id}: {threshold:.6f}")

## Single Threshold Evaluation

In [ ]:
results = sg.evaluation(
    input_data=X_test,
    target_col=y_test,
    threshold_ratio=0.75,  # Adjusts the anomaly detection sensitivity relative to baseline threshold
)

preds_test = sg.preds_test
recon_metrics_test = sg.reconstruction_test

print("Single Threshold Evaluation Results:")
print(results['global metrics'])

## Multiple Threshold Optimization

Test different threshold ratios to find the optimal configuration.

In [ ]:
threshold_ratios = [0.75, 1, 1.5, 2, 3, 4]

global_results = pd.concat(
    [
        sg.evaluation(input_data=X_test, target_col=y_test, threshold_ratio=thr)["global metrics"]
        for thr in threshold_ratios
    ]
)

print("\nThreshold Optimization Results:")
print(global_results[['Threshold Ratio', 'Accuracy', 'F1 Score', 'Precision', 'Recall']])

## Final Anomaly Detection

Apply the best threshold configuration to detect anomalies in future data.

In [ ]:
best_ratio = global_results.sort_values(by="Accuracy", ascending=False).iloc[0]["Threshold Ratio"]

predictions = sg.detections(input_data=X_future_data, threshold_ratio=best_ratio)

final_metrics = metrics_classification(
    y_true=y_future_data, y_pred=predictions["Predicted Anomalies"]
)

print(f"\nBest threshold ratio: {best_ratio}")
print("\nFinal Performance Metrics:")
for metric in final_metrics.columns:
    print(f"  {metric}: {final_metrics[metric].iloc[0]:.4f}")

print("\n Detection Summary:")
print(f"  Anomalies detected: {predictions['Predicted Anomalies'].sum()}")
print(f"  Total samples: {len(predictions)}")
print(f"  Anomaly rate: {predictions['Predicted Anomalies'].mean():.2%}")

# Every row comes with its verdict, cluster, anomaly score and threshold
predictions[["Predicted Anomalies", "Cluster", "Reconstruction Error", "Threshold"]].head()